In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 25.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 25.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [3]:
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

dagshub.init(repo_owner='Nestor-Dzadzamia', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=91416701-20c7-413b-9412-f8fc7f5b67f5&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=54e0fd558d7921ea91f025e8ca3af923d68a2c771025611ece9fbb78ad988d7b




Accessing as Nestor-Dzadzamia

Initialized MLflow to track repo "Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection"

Repository Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection initialized!

In [4]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

print(train_transaction.shape)
print(train_identity.shape)

(590540, 394)
(144233, 41)


# Merge data then split as train test

In [5]:
df = train_transaction.merge(train_identity, on='TransactionID', how='left')
print(f"Merged shape: {df.shape}")

y = df['isFraud']
X = df.drop(columns=['isFraud', 'TransactionID'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Fraud ratio: {y_train.mean():.4f}")

Merged shape: (590540, 434)
X_train shape: (472432, 432)
X_test shape: (118108, 432)
Fraud ratio: 0.0350


# Cleaning, Feature Engineering & Feature Selection

In [6]:
mlflow.set_experiment("XGBoost_Training")

# Cleaning run
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    cols_dropped = X.columns[X.isnull().mean() > 0.7].tolist()
    mlflow.log_param("null_thresh", 0.7)
    mlflow.log_param("num_imputer_strategy", "median")
    mlflow.log_param("cat_imputer_strategy", "most_frequent")
    mlflow.log_metric("cols_dropped", len(cols_dropped))
    mlflow.log_metric("cols_remaining", X.shape[1] - len(cols_dropped))

# Feature Engineering run
with mlflow.start_run(run_name="XGBoost_Feature_Engineering"):
    mlflow.log_param("new_features", "log_TransactionAmt, Transaction_hour, Transaction_day, TransactionAmt_freq, card1_freq")
    mlflow.log_metric("original_cols", X.shape[1])
    mlflow.log_metric("new_cols_added", 5)

# Feature Selection run
with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    mlflow.log_param("corr_thresh", 0.9)
    mlflow.log_param("iv_thresh", 0.02)
    mlflow.log_param("methods", "correlation_filter + WOE_IV")

print("All preprocessing runs logged!")

🏃 View run XGBoost_Cleaning at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/4fd8f5988bcb4ba5ac7a37ea548ba8bc
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4
🏃 View run XGBoost_Feature_Engineering at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/6d55379ca93f416eab00bebc3f4b1707
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4
🏃 View run XGBoost_Feature_Selection at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/ab6c0fbe032f4f339a589bbcdcdb7c6b
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4
All preprocessing runs logged!


# Creating the preprocessor class

In [24]:
class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, null_thresh=0.5, corr_thresh=0.9, iv_thresh=0.02):
        self.null_thresh = null_thresh
        self.corr_thresh = corr_thresh
        self.iv_thresh = iv_thresh

    def _feature_engineering(self, X):
        X = X.copy()
        # Basic features
        X['log_TransactionAmt'] = np.log1p(X['TransactionAmt'])
        X['Transaction_hour'] = (X['TransactionDT'] // 3600) % 24
        X['Transaction_day'] = (X['TransactionDT'] // (3600 * 24)) % 7
        # Amount frequency encoding
        amt_freq = X['TransactionAmt'].value_counts(normalize=True)
        X['TransactionAmt_freq'] = X['TransactionAmt'].map(amt_freq)
        # Card1 frequency encoding
        card1_freq = X['card1'].value_counts(normalize=True)
        X['card1_freq'] = X['card1'].map(card1_freq)
        return X

    def _calculate_iv(self, X, y):
        iv_scores = {}
        for col in X.columns:
            try:
                temp = pd.DataFrame({'feature': X[col], 'target': y})
                temp['feature'] = temp['feature'].fillna('missing').astype(str)
                grouped = temp.groupby('feature')['target'].agg(['sum', 'count'])
                grouped.columns = ['events', 'total']
                grouped['non_events'] = grouped['total'] - grouped['events']
                total_events = grouped['events'].sum()
                total_non_events = grouped['non_events'].sum()
                grouped['dist_events'] = grouped['events'] / (total_events + 1e-10)
                grouped['dist_non_events'] = grouped['non_events'] / (total_non_events + 1e-10)
                grouped['woe'] = np.log((grouped['dist_events'] + 1e-10) / (grouped['dist_non_events'] + 1e-10))
                grouped['iv'] = (grouped['dist_events'] - grouped['dist_non_events']) * grouped['woe']
                iv_scores[col] = grouped['iv'].sum()
            except:
                iv_scores[col] = 0
        return iv_scores

    def fit(self, X, y=None):
        X = self._feature_engineering(X)
        # Drop high null columns
        self.cols_to_drop_ = X.columns[X.isnull().mean() > self.null_thresh].tolist()
        X = X.drop(columns=self.cols_to_drop_)
        # Correlation filter
        corr_matrix = X.select_dtypes(include=np.number).corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.corr_cols_to_drop_ = [col for col in upper.columns if any(upper[col] > self.corr_thresh)]
        X = X.drop(columns=self.corr_cols_to_drop_)
        # WOE/IV filter
        if y is not None:
            iv_scores = self._calculate_iv(X, y)
            self.low_iv_cols_ = [col for col, iv in iv_scores.items() if iv < self.iv_thresh]
            self.iv_scores_ = iv_scores
        else:
            self.low_iv_cols_ = []
            self.iv_scores_ = {}
        X = X.drop(columns=self.low_iv_cols_)
        # Cat/num cols
        self.cat_cols_ = [col for col in X.columns if X[col].dtype == 'object']
        self.num_cols_ = [col for col in X.columns if X[col].dtype != 'object']
        # Fit imputers
        self.num_imputer_ = SimpleImputer(strategy='median')
        self.num_imputer_.fit(X[self.num_cols_])
        self.cat_imputer_ = SimpleImputer(strategy='most_frequent')
        self.cat_imputer_.fit(X[self.cat_cols_])
        # Fit encoder
        self.encoder_ = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        X_cat_imputed = self.cat_imputer_.transform(X[self.cat_cols_])
        self.encoder_.fit(X_cat_imputed)
        # Fit scaler
        self.scaler_ = StandardScaler()
        X_num_imputed = self.num_imputer_.transform(X[self.num_cols_])
        self.scaler_.fit(X_num_imputed)
        return self

    def transform(self, X):
        X = self._feature_engineering(X)
        X = X.drop(columns=self.cols_to_drop_, errors='ignore')
        X = X.drop(columns=self.corr_cols_to_drop_, errors='ignore')
        X = X.drop(columns=self.low_iv_cols_, errors='ignore')
        X_num = self.num_imputer_.transform(X[self.num_cols_])
        X_cat = self.cat_imputer_.transform(X[self.cat_cols_])
        X_cat_encoded = self.encoder_.transform(X_cat)
        X_num_scaled = self.scaler_.transform(X_num)
        X_final = np.hstack([X_num_scaled, X_cat_encoded])
        return X_final

# Pipeline and experiments

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, precision_recall_curve, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

mlflow.set_experiment("XGBoost_Training")

# Calculate scale_pos_weight for imbalance
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

experiments = [
    {"null_thresh": 0.5, "corr_thresh": 0.9, "iv_thresh": 0.02, "n_estimators": 100, "max_depth": 5,  "learning_rate": 0.1,  "subsample": 0.8},
    {"null_thresh": 0.5, "corr_thresh": 0.9, "iv_thresh": 0.02, "n_estimators": 200, "max_depth": 5,  "learning_rate": 0.05, "subsample": 0.8},
    {"null_thresh": 0.7, "corr_thresh": 0.9, "iv_thresh": 0.02, "n_estimators": 200, "max_depth": 6,  "learning_rate": 0.1,  "subsample": 0.8},
    {"null_thresh": 0.7, "corr_thresh": 0.9, "iv_thresh": 0.01, "n_estimators": 200, "max_depth": 6,  "learning_rate": 0.1,  "subsample": 0.9},
    {"null_thresh": 0.7, "corr_thresh": 0.9, "iv_thresh": 0.02, "n_estimators": 300, "max_depth": 6,  "learning_rate": 0.05, "subsample": 0.9},
    {"null_thresh": 0.7, "corr_thresh": 0.9, "iv_thresh": 0.02, "n_estimators": 300, "max_depth": 8,  "learning_rate": 0.05, "subsample": 0.9},
]

for exp in experiments:
    pipe = Pipeline(steps=[
        ('preprocessor', FraudPreprocessor(
            null_thresh=exp["null_thresh"],
            corr_thresh=exp["corr_thresh"],
            iv_thresh=exp["iv_thresh"]
        )),
        ('model', XGBClassifier(
            n_estimators=exp["n_estimators"],
            max_depth=exp["max_depth"],
            learning_rate=exp["learning_rate"],
            subsample=exp["subsample"],
            scale_pos_weight=scale_pos_weight,
            tree_method='hist',
            objective='binary:logistic',
            eval_metric='auc',
            random_state=42
        ))
    ])

    pipe.fit(X_train, y_train)

    with mlflow.start_run(run_name=f"XGB_n={exp['n_estimators']}_depth={exp['max_depth']}_lr={exp['learning_rate']}_null={exp['null_thresh']}"):
        mlflow.log_params({**exp, "scale_pos_weight": scale_pos_weight, "tree_method": "hist"})

        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        y_train_proba = pipe.predict_proba(X_train)[:, 1]

        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_proba)
        mlflow.log_metric("train_roc_auc", train_roc_auc)
        mlflow.log_metric("test_roc_auc", test_roc_auc)

        # Classification report
        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric_name}", value)

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        mlflow.log_metric("tn", cm[0,0])
        mlflow.log_metric("fp", cm[0,1])
        mlflow.log_metric("fn", cm[1,0])
        mlflow.log_metric("tp", cm[1,1])

        # Threshold tuning
        precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
        try:
            idx = np.where(recalls >= 0.75)[0][0]
            optimal_threshold = thresholds[idx]
        except IndexError:
            optimal_threshold = 0.5
        y_pred_adj = (y_proba >= optimal_threshold).astype(int)
        mlflow.log_metric("optimal_threshold", optimal_threshold)
        mlflow.log_metric("adj_f1", f1_score(y_test, y_pred_adj))
        mlflow.log_metric("adj_precision", precision_score(y_test, y_pred_adj))
        mlflow.log_metric("adj_recall", recall_score(y_test, y_pred_adj))

        # Cross validation
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='roc_auc')
        mlflow.log_metric("cv_mean_roc_auc", cv_scores.mean())
        mlflow.log_metric("cv_std_roc_auc", cv_scores.std())

        # Save pipeline
        mlflow.sklearn.log_model(pipe, artifact_path="xgboost_pipeline")

        print(f"XGB n={exp['n_estimators']} depth={exp['max_depth']} lr={exp['learning_rate']} null={exp['null_thresh']} → Train: {train_roc_auc:.4f} Test: {test_roc_auc:.4f} CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | Optimal threshold: {optimal_threshold:.4f}")

# Register the best model

In [11]:
import mlflow

best_run_id = "50fea963a8af4bd085fe64241b85ce86"

model_uri = f"runs:/{best_run_id}/xgboost_pipeline"


mlflow.set_experiment("XGBoost_Training")

with mlflow.start_run(run_id=best_run_id):
    mlflow.sklearn.log_model(
        pipe,
        name="IEEE_Fraud_Detection_Best_Model",
        registered_model_name="IEEE_Fraud_Detection_Best_Model"
    )

print("Model registered!")

2026/05/07 21:05:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'IEEE_Fraud_Detection_Best_Model' already exists. Creating a new version of this model...
2026/05/07 21:06:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IEEE_Fraud_Detection_Best_Model, version 1
Created version '1' of model 'IEEE_Fraud_Detection_Best_Model'.


🏃 View run XGB_n=300_depth=8_lr=0.05_null=0.7 at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/50fea963a8af4bd085fe64241b85ce86
🧪 View experiment at: https://dagshub.com/Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4
Model registered!
